# ROPAgen document eval

In my Master Thesis "ROPAgen", I created a web app which lets users create ROPA documents easily for GDPR purposes.

ROPAgen is powered by AI: There are three modes of LLM integration:

Form: Users can tick checkboxes to create a ROPA document. An "AI Suggest" button fills out the rest of the checkboxes for the user based on already given input (most user input).

Ask: The user uses an AI chatbot and ask questions about the subsection of the ROPA and what they might need to add to the document in String format. The user has to give the final answer themselves (medium user input).

Chat: Similar to Ask, but instead of the user having to input the final data, the AI does it on behalf of the user (lest user input).


Users were tasked to create three separate documents with all three modes.
After having done a user survery for the usabilty and credibilty of ROPAgen, we want to evaluate the documents that users generated.



## How to evaluate

We are going to evaluate all documents that were saved to the DB.
Every user had the same task and requirements for each document.

Evaluation will be done twofold:
1. BERT score using a reference document
2. LLM evaluaiton (any LLM other than Mistral)

## Document task:

Aufgabe:
Bitte verwenden Sie dieses Szenario als Grundlage, um im jeweiligen Modus in ROPAgen ein ROPA-Dokument zu erstellen und anschließend zu speichern.


Szenario: Mitarbeiterverwaltung (Arbeitszeit/Urlaub) & Parkplatzberechtigung
Sie sind Mitarbeitender namens A.I. und arbeiten für die APOR GmbH, ein Softwareunternehmen mit ca. 50 Mitarbeitenden in Ulm. Sie handeln im Namen des Unternehmens und sind im Szenario sowohl Verantwortlicher (Data Controller) als auch Datenschutzbeauftragte:r (DPO/DSB).

Zur Organisation :

Rolle: Data Controller, DPO

Adresse: Mainstraße 67, Ulm

E-Mail: mail@apor.eu

Telefon: 012345678

Vertreter: (Sie), A. I., ai@apor.eu

DPO: Das sind auch Sie


Verarbeitung
Das Unternehmen verarbeitet personenbezogene Daten, um Mitarbeitende vom Eintritt bis zum Austritt zu verwalten. Im Fokus stehen:

Anlegen neuer Mitarbeitender

Dokumentation von Arbeitszeiten

Verwaltung von Urlaub und Abwesenheiten

Zusätzlich verwaltet APOR für berechtigte Mitarbeitende eine Parkplatzberechtigung für einen Firmenparkplatz mit Schranke. Damit berechtigte Mitarbeitende den Parkplatz nutzen können, wird ein geeignetes Merkmal zur Zuordnung der Berechtigung hinterlegt (z. B. ein Fahrzeugkennzeichen).
APOR arbeitet mit intern gehostetem Microsoft Office sowie internen Servern (keine externe Cloud). Krankentage oder medizinische Angaben sind für diese Verarbeitung nicht erforderlich.


Für die Verwaltung sind technische und organisatorische Maßnahmen vorgesehen, wie z.B. Zugriffsbeschränkung nach Rollen (Need-to-know), Passwortschutz (ggf. MFA), regelmäßige Backups, Zutrittskontrolle zu Serverraum/Archiv und Vertraulichkeitsverpflichtung der berechtigten Mitarbeitenden.

Alle in diesem Zusammenhang gespeicherten Informationen werden spätestens 3 Monate nach Austritt der jeweiligen Person gelöscht.

In [ ]:
import pandas as pd
import numpy as np
from bert_score import score
import os
import torch

# Device detection for VRAM usage
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running BERTScore on: {device}')

ref_path = "./123_form.txt"
data_path = "./documents.csv"

with open(ref_path, "r") as f:
    reference_text = f.read()

full_df = pd.read_csv(data_path).dropna(subset=["ai_mode", "document"])

def evaluate_subset(subset_df, label):
    if subset_df.empty:
        return None, None

    print(f"Evaluating {label} documents using microsoft/deberta-xlarge-mnli on {device}...")
    candidates = subset_df["document"].tolist()
    refs = [reference_text] * len(candidates)

    # The library automatically uses the 'device' if passed or detected
    P, R, F1 = score(candidates, refs, model_type="microsoft/deberta-xlarge-mnli", lang="de", device=device, verbose=True)

    subset_df = subset_df.copy()
    subset_df["bert_precision"] = P.tolist()
    subset_df["bert_recall"] = R.tolist()
    subset_df["bert_f1"] = F1.tolist()

    stats = {"Mode": label}
    for metric, col in [("P", "bert_precision"), ("R", "bert_recall"), ("F1", "bert_f1")]:
        stats[f"Mean_{metric}"] = subset_df[col].mean()
        stats[f"Median_{metric}"] = subset_df[col].median()
        stats[f"Lowest_{metric}"] = subset_df[col].min()
        stats[f"Highest_{metric}"] = subset_df[col].max()

    return subset_df, stats

modes = ["form", "ask", "chat"]
processed_dfs = []
stats_list = []

for m in modes:
    m_df, m_stats = evaluate_subset(full_df[full_df["ai_mode"] == m], m)
    if m_df is not None:
        processed_dfs.append(m_df)
        stats_list.append(m_stats)

if processed_dfs:
    combined_df = pd.concat(processed_dfs)
    stats_df = pd.DataFrame(stats_list)
    display(stats_df)
else:
    print("No data found to evaluate.")

Running BERTScore on: cuda
Evaluating form documents using microsoft/deberta-xlarge-mnli on cuda...


pytorch_model.bin:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

### Understanding BERTScore Metrics

Unlike traditional BLEU/ROUGE scores that look for exact word overlaps, BERTScore uses contextual embeddings to find semantic similarity.

*   **Precision (P):** Measures how much of the user-generated document is semantically present in the reference. High precision means the AI didn't add much 'irrelevant' content.
*   **Recall (R):** Measures how much of the reference document is semantically present in the user-generated one. High recall means the user successfully captured all the requirements from the task scenario.
*   **F1 Score:** The harmonic mean of P and R. It is the most robust 'overall' metric because a document that is very short (high precision but low recall) or a document that is just a wall of text (high recall but low precision) will both result in a lower F1 score.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

if 'stats_df' in globals() and 'combined_df' in globals():
    mode_order = ['form', 'ask', 'chat']
    metrics = [('Mean_P', 'Precision'), ('Mean_R', 'Recall'), ('Mean_F1', 'F1')]

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    for i, (col, title) in enumerate(metrics):
        sns.barplot(x="Mode", y=col, data=stats_df, order=mode_order, ax=axes[i], palette="viridis", hue="Mode", legend=False)
        axes[i].set_title(f"Average {title} per Mode")
        axes[i].set_ylim(0, 1)
        for p in axes[i].patches:
            axes[i].annotate(f'{p.get_height():.4f}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='center', xytext=(0, 9), textcoords='offset points')

    plt.tight_layout()
    plt.show()

    # Boxplot for distribution of all metrics
    melted_df = combined_df.melt(id_vars=['ai_mode'], value_vars=['bert_precision', 'bert_recall', 'bert_f1'], var_name='Metric', value_name='Score')
    plt.figure(figsize=(14, 7))
    sns.boxplot(x="ai_mode", y="Score", hue="Metric", data=melted_df, order=mode_order)
    plt.title("Distribution of Precision, Recall, and F1 across Modes")
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.show()
else:
    print("Dataframes not found.")